In [2]:
import torch
import torch.nn as nn

from transformers.cache_utils import DynamicCache
from transformers.models.llama.modeling_llama import (
    LlamaDecoderLayer,
    LlamaRMSNorm,
    LlamaRotaryEmbedding,
)


class Model1(nn.Module):
    def __init__(self, config, use_cache: bool = False):
        super().__init__()

        self.use_cache = use_cache
        self.config = config
        self.config._attn_implementation = "spda"
        self.embed_tokens = nn.Embedding(
            config.vocab_size,
            config.hidden_size,
            padding_idx=config.pad_token_id,
        )

        self.rotary_emb = LlamaRotaryEmbedding(config=config)

        self.layers = nn.ModuleList([
            LlamaDecoderLayer(config, layer_idx=i)
            for i in range(14)
        ])

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
        past_key_values: DynamicCache | None = None,
        cache_position: torch.Tensor | None = None,
        use_cache: bool | None = None,
    ):
        if use_cache is None:
            use_cache = self.use_cache

        device = input_ids.device
        batch_size, seq_len = input_ids.shape

        hidden_states = self.embed_tokens(input_ids)

        if cache_position is not None:
            position_ids = cache_position.unsqueeze(0).expand(batch_size, -1)
        elif attention_mask is not None:
            position_ids = attention_mask.cumsum(dim=-1) - 1
            position_ids = position_ids.masked_fill(attention_mask == 0, 0)
        else:
            position_ids = torch.arange(seq_len, device=device).unsqueeze(0)
            position_ids = position_ids.expand(batch_size, -1)

        if use_cache and past_key_values is None:
            past_key_values = DynamicCache()

        position_embeddings = self.rotary_emb(hidden_states, position_ids)

        for layer in self.layers:
            layer_outputs = layer(
                hidden_states,
                attention_mask=None,
                position_ids=position_ids,
                past_key_value=past_key_values,
                cache_position=cache_position,
                position_embeddings=position_embeddings,
                use_cache=use_cache,
            )
            hidden_states = layer_outputs

        return {
            "hidden_states": hidden_states,
            "past_key_values": past_key_values,
        }

/Users/hongkwon/miniconda3/envs/llama310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class Model2(nn.Module):
    def __init__(self, config, use_cache: bool = False):
        super().__init__()

        self.use_cache = use_cache
        self.config = config
        self.config._attn_implementation = "spda"
        self.rotary_emb = LlamaRotaryEmbedding(config=config)

        self.layers = nn.ModuleList([
            LlamaDecoderLayer(config, layer_idx=i)
            for i in range(14, config.num_hidden_layers)
        ])

        self.norm = LlamaRMSNorm(
            config.hidden_size,
            eps=config.rms_norm_eps,
        )

        self.lm_head = nn.Linear(
            config.hidden_size,
            config.vocab_size,
            bias=False,
        )

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
        past_key_values: DynamicCache | None = None,
        cache_position: torch.Tensor | None = None,
        use_cache: bool | None = None,
    ):
        if use_cache is None:
            use_cache = self.use_cache

        device = hidden_states.device
        batch_size, seq_len, _ = hidden_states.shape

        if cache_position is not None:
            position_ids = cache_position.unsqueeze(0).expand(batch_size, -1)
        elif attention_mask is not None:
            position_ids = attention_mask.cumsum(dim=-1) - 1
            position_ids = position_ids.masked_fill(attention_mask == 0, 0)
        else:
            position_ids = torch.arange(seq_len, device=device).unsqueeze(0)
            position_ids = position_ids.expand(batch_size, -1)

        if use_cache and past_key_values is None:
            past_key_values = DynamicCache()

        position_embeddings = self.rotary_emb(hidden_states, position_ids)

        for layer in self.layers:
            layer_outputs = layer(
                hidden_states,
                attention_mask=None,
                position_ids=position_ids,
                past_key_value=past_key_values,
                cache_position=cache_position,
                position_embeddings=position_embeddings,
                use_cache=use_cache,
            )
            hidden_states = layer_outputs

        hidden_states = self.norm(hidden_states)
        logits = self.lm_head(hidden_states)

        return {
            "logits": logits,
            "hidden_states": hidden_states,
            "past_key_values": past_key_values,
        }

In [4]:
from transformers import AutoConfig
from safetensors.torch import load_file
MODEL_PATH = "models/Llama-3.2-3B-Instruct-split"

config = AutoConfig.from_pretrained(MODEL_PATH)

model1 = Model1(config, use_cache=True)

model2 = Model2(config, use_cache=True)
state1 = load_file("models/Llama-3.2-3B-Instruct-split/model1.safetensors")

model1.load_state_dict(state1, strict=True)

state2 = load_file("models/Llama-3.2-3B-Instruct-split/model2.safetensors")

model2.load_state_dict(state2)

<All keys matched successfully>

In [5]:
from transformers import AutoTokenizer
device = "cpu"
prompt = "KV cache가 뭐야? 짧게 대답해줘."
tokenizer = AutoTokenizer.from_pretrained("models/Llama-3.2-3B-Instruct-split")

inputs = tokenizer(prompt, return_tensors="pt")

attention_mask = inputs.get("attention_mask", None)

if attention_mask is not None:
    attention_mask = attention_mask.to(device)




In [6]:
input_ids = inputs['input_ids'].to(device)

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
from transformers import AutoTokenizer, AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "models/Llama-3.2-3B-Instruct",
    torch_dtype=torch.float32,
).to(device)

model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards:   0%|          | 0/2 [00:09<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
model.config._attn_implementation

'sdpa'

In [ ]:
# model1.config._attn_implementation = 'sdpa'
# model2.config._attn_implementation = 'sdpa'

In [ ]:
# model.eval()
model1.eval()
model2.eval()

generated_ref = input_ids.clone()
generated_pipe = input_ids.clone()

past_ref = None
past1 = None
past2 = None

cache_len = input_ids.shape[1]

with torch.no_grad():
    for step in range(10):
        if step == 0:
            ref_input = input_ids
            pipe_input = input_ids
            cache_position = torch.arange(input_ids.shape[1], device=input_ids.device)
        else:
            ref_input = ref_next
            pipe_input = pipe_next
            cache_position = torch.tensor([cache_len], device=input_ids.device)

        # 원본도 cache decode
        ref_out = model(
            input_ids=ref_input,
            past_key_values=past_ref,
            cache_position=cache_position,
            use_cache=True,
        )
        past_ref = ref_out.past_key_values
        print(ref_out['logits'].shape)
        ref_next = ref_out.logits[:, -1].argmax(dim=-1, keepdim=True)

        # pipeline cache decode
        out1 = model1(
            input_ids=pipe_input,
            past_key_values=past1,
            cache_position=cache_position,
            use_cache=True,
        )
        past1 = out1["past_key_values"]

        out2 = model2(
            hidden_states=out1["hidden_states"],
            past_key_values=past2,
            cache_position=cache_position,
            use_cache=True,
        )
        past2 = out2["past_key_values"]

        pipe_next = out2["logits"][:, -1].argmax(dim=-1, keepdim=True)

        generated_ref = torch.cat([generated_ref, ref_next], dim=-1)
        generated_pipe = torch.cat([generated_pipe, pipe_next], dim=-1)

        print(f"\nSTEP {step}")
        print("ref token :", ref_next.item(), tokenizer.decode(ref_next[0]))
        print("pipe token:", pipe_next.item(), tokenizer.decode(pipe_next[0]))
        print("same:", torch.equal(ref_next, pipe_next))
        cache_len += 1

NameError: name 'model1' is not defined

In [ ]:
past1[0][0].shape

torch.Size([1, 8, 15, 128])

In [ ]:
from transformers import AutoTokenizer
device = "cuda"
prompt = "what's Jansports"
tokenizer = AutoTokenizer.from_pretrained("models/Llama-3.2-3B-Instruct-split")

inputs = tokenizer(prompt, return_tensors="pt")

attention_mask = inputs.get("attention_mask", None)

if attention_mask is not None:
    attention_mask = attention_mask.to(device)
input_ids = inputs['input_ids'].to(device)

In [ ]:
# model.eval()
model1.eval()
model2.eval()

# generated_ref = input_ids.clone()
generated_pipe = input_ids.clone()
eos_id = tokenizer.eos_token_id
result = ""
# past_ref = None
past1 = None
past2 = None
model1 = model1.to(device)
model2 = model2.to(device)
cache_len = input_ids.shape[1]

with torch.no_grad():
    for step in range(1000):
        if step == 0:
            # ref_input = input_ids
            pipe_input = input_ids
            cache_position = torch.arange(input_ids.shape[1], device=input_ids.device)
        else:
            # ref_input = ref_next
            pipe_input = pipe_next
            cache_position = torch.tensor([cache_len], device=input_ids.device)

        # 원본도 cache decode
        # ref_out = model(
        #     input_ids=ref_input,
        #     past_key_values=past_ref,
        #     cache_position=cache_position,
        #     use_cache=True,
        # )
        # past_ref = ref_out.past_key_values
        # ref_next = ref_out.logits[:, -1].argmax(dim=-1, keepdim=True)

        # pipeline cache decode
        out1 = model1(
            input_ids=pipe_input,
            past_key_values=past1,
            cache_position=cache_position,
            use_cache=True,
        )
        past1 = out1["past_key_values"]

        out2 = model2(
            hidden_states=out1["hidden_states"],
            past_key_values=past2,
            cache_position=cache_position,
            use_cache=True,
        )
        past2 = out2["past_key_values"]

        pipe_next = out2["logits"][:, -1].argmax(dim=-1, keepdim=True)

        # generated_ref = torch.cat([generated_ref, ref_next], dim=-1)
        generated_pipe = torch.cat([generated_pipe, pipe_next], dim=-1)

        # print(f"\nSTEP {step}")
        # print("ref token :", ref_next.item(), tokenizer.decode(ref_next[0]))
        # print("pipe token:", pipe_next.item(), tokenizer.decode(pipe_next[0]))
        if pipe_next.item() == eos_id:
            break
        result += tokenizer.decode(pipe_next[0])
        # print("same:", torch.equal(ref_next, pipe_next))
        
        cache_len += 1
        print(tokenizer.decode(pipe_next[0]), end="")

/home/eslab/anaconda3/envs/llama310/lib/python3.12/site-packages/torch/nn/modules/module.py:1787: FutureWarning: `past_key_value` is deprecated and will be removed in version 4.58 for `LlamaDecoderLayer.forward`. Use `past_key_values` instead.
  return forward_call(*args, **kwargs)


?
JanSport is a brand of backpacks, bags, and other school supplies that is owned by JanSport, Inc., a subsidiary of VF Corporation. JanSport was founded in 1967 by Janie and Joe Roth in San Francisco, California. The company is known for its high-quality, durable, and stylish backpacks and bags that are popular among students, travelers, and outdoor enthusiasts.

JanSport products are designed to be functional, comfortable, and fashionable, with a focus on providing excellent value for the price. The company offers a wide range of products, including backpacks, messenger bags, tote bags, lunch boxes, and more. JanSport products are available at many retail stores, online marketplaces, and the company's own website.

JanSport has become a well-known and respected brand in the school supply industry, and its products are popular among students, teachers, and parents alike. The company's commitment to quality, durability, and style has helped it to build a loyal customer base over the ye

In [ ]:

inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs["input_ids"].to(device)

attention_mask = inputs.get("attention_mask", None)

if attention_mask is not None:
    attention_mask = attention_mask.to(device)



model.eval()
model1.eval()
model2.eval()



with torch.no_grad():
    # 1. 원본 모델
    ref = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
    )
    ref_logits = ref.logits

    # 2. 쪼갠 모델
    out1 = model1(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
    )

    out2 = model2(
        hidden_states=out1["hidden_states"],
        attention_mask=attention_mask,
        use_cache=False,
    )

    pipe_logits = out2["logits"]

    # 3. 비교
    diff = (ref_logits - pipe_logits).abs()

    print("ref logits shape :", ref_logits.shape)
    print("pipe logits shape:", pipe_logits.shape)

    print("max diff :", diff.max().item())
    print("mean diff:", diff.mean().item())

    ref_next = ref_logits[:, -1].argmax(dim=-1, keepdim=True)
    pipe_next = pipe_logits[:, -1].argmax(dim=-1, keepdim=True)

    print("ref next token id :", ref_next.item())
    print("pipe next token id:", pipe_next.item())

    print("ref next token :", tokenizer.decode(ref_next[0]))
    print("pipe next token:", tokenizer.decode(pipe_next[0]))

RuntimeError: Expected all tensors to be on the same device, but got index is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA__index_select)

In [ ]:
import torch
torch.empty([1,2]).shape
ls = {'a':2, 'b':3, 'c':3}
len(ls)
list(ls.values())

[2, 3, 3]

In [ ]:
import inspect
def func(x, y, se):
    return x + y


inspect.signature(func).parameters

mappingproxy({'x': <Parameter "x">,
              'y': <Parameter "y">,
              'se': <Parameter "se">})